# Practical 6A — Pascal VOC R-CNN Family

## Aim
Explore Pascal VOC and demonstrate region-proposal-based object detection using a pretrained Faster R-CNN model.

## Setup
Run the installation cell if required. These notebooks use Python 3.10–3.12 where possible. On Apple Silicon, TensorFlow/PyTorch acceleration depends on the installed build; CPU execution is supported.

**Important:** dataset downloads and model training may take time. Results are generated when you run the notebook; no accuracy values are pre-filled.

In [ ]:
%pip install -q numpy pandas matplotlib scikit-learn tensorflow

## Requirements\nPyTorch and torchvision. First run downloads VOC 2007 validation images and pretrained model weights. This is inference-oriented; full training requires a larger compute budget.

## 1. Environment and Pascal VOC dataset\nThis practical demonstrates the two-stage R-CNN family. Full VOC training is compute- and storage-intensive. The notebook uses torchvision's pretrained Faster R-CNN detector as a practical region-proposal-based R-CNN-family implementation. Classic R-CNN and Fast R-CNN differ in proposal feature extraction and training pipelines; the distinctions are documented below.

In [ ]:
%pip install -q torch torchvision pillow matplotlib

## 2. Download a small Pascal VOC dataset through torchvision

In [ ]:
import torch, torchvision
from torchvision.datasets import VOCDetection
from torchvision.transforms import functional as F
from torch.utils.data import DataLoader
from PIL import Image
import matplotlib.pyplot as plt
dataset=VOCDetection(root="./data",year="2007",image_set="val",download=True)
print("VOC validation images:",len(dataset))

## 3. Inspect a Pascal VOC sample and annotations

In [ ]:
img,ann=dataset[0]
plt.imshow(img); plt.axis("off"); plt.show()
print("Annotation keys:",ann["annotation"].keys())
print("Objects:",ann["annotation"].get("object",[])[:2])

## 4. Load pretrained Faster R-CNN (region-proposal-based detector)

In [ ]:
from torchvision.models.detection import fasterrcnn_resnet50_fpn, FasterRCNN_ResNet50_FPN_Weights
weights=FasterRCNN_ResNet50_FPN_Weights.DEFAULT
detector=fasterrcnn_resnet50_fpn(weights=weights).eval()
device="mps" if torch.backends.mps.is_available() else "cpu"
detector.to(device)

## 5. Run inference and visualize bounding boxes

In [ ]:
from torchvision.utils import draw_bounding_boxes
from torchvision.transforms.functional import to_tensor, to_pil_image
image=dataset[0][0]
tensor=to_tensor(image).to(device)
with torch.no_grad():
    prediction=detector([tensor])[0]
keep=prediction["scores"].cpu()>0.5
boxes=prediction["boxes"].cpu()[keep]
labels=prediction["labels"].cpu()[keep]
scores=prediction["scores"].cpu()[keep]
class_names=weights.meta["categories"]
label_text=[f"{class_names[int(l)]}: {float(s):.2f}" for l,s in zip(labels,scores)]
vis=draw_bounding_boxes((tensor.cpu()*255).byte(),boxes,labels=label_text,width=2)
plt.figure(figsize=(10,8)); plt.imshow(to_pil_image(vis)); plt.axis("off"); plt.show()

## 6. R-CNN vs Fast R-CNN vs Faster R-CNN

| Method | Main idea |
|---|---|
| R-CNN | Generates region proposals, runs a CNN separately on each region, then classifies regions. |
| Fast R-CNN | Computes a shared convolutional feature map and uses RoI pooling for proposals. |
| Faster R-CNN | Adds a Region Proposal Network (RPN) to generate proposals within the network. |

**Important:** this notebook runs pretrained Faster R-CNN inference on VOC images; it does not train original R-CNN and Fast R-CNN independently. This is a feasible demonstration of the region-based detector family.

## 7. Conclusion\nReview predicted classes, confidence scores and bounding boxes. Detection performance depends on class coverage, image quality, confidence threshold and dataset/domain match.